# ipywidgets `interact` — reference example

This notebook shows the classic `ipywidgets.interact()` pattern:
define a typed function, get controls for free.

**dash-fn-form** brings this same workflow to Plotly Dash.

```
ipywidgets          →   dash-fn-form
────────────────────────────────────────
interact(fn)        →   build_config(id, fn)
auto widgets        →   auto dcc components
live update         →   callback + Apply
Jupyter only        →   any Dash app
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ipywidgets import interact

## The function

Just a regular Python function with typed, defaulted parameters.
No UI code whatsoever.

In [2]:
def make_wave(
    amplitude: float = 1.0,
    frequency: float = 1.0,
    phase: float = 0.0,
    n_cycles: int = 3,
    color: str = "blue",
    show_envelope: bool = False,
):
    """A simple sine wave with configurable parameters."""
    x = np.linspace(0, n_cycles * 2 * np.pi, 500)
    y = amplitude * np.sin(frequency * x + phase)

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(x, y, color=color)
    if show_envelope:
        ax.axhline(amplitude, color=color, linestyle=":", linewidth=1)
        ax.axhline(-amplitude, color=color, linestyle=":", linewidth=1)
    ax.set_ylim(-2.5, 2.5)
    ax.set_xlabel("x")
    plt.tight_layout()
    plt.show()

## `interact()` — one call, full UI

ipywidgets inspects the function signature and default values to
auto-generate sliders, text inputs, checkboxes, and dropdowns.
The function is called live on every change.

In [3]:
interact(
    make_wave,
    amplitude=(0.1, 2.5, 0.1),
    frequency=(0.1, 5.0, 0.1),
    phase=(-np.pi, np.pi, 0.1),
    n_cycles=(1, 10, 1),
    color=["blue", "red", "green", "orange"],
    show_envelope=False,
)

interactive(children=(FloatSlider(value=1.0, description='amplitude', max=2.5, min=0.1), FloatSlider(value=1.0…

<function __main__.make_wave(amplitude: float = 1.0, frequency: float = 1.0, phase: float = 0.0, n_cycles: int = 3, color: str = 'blue', show_envelope: bool = False)>

## `interact` with pure type-hint inference

With no hints to `interact()`, it falls back to the default values alone
to infer widget type — `float` → `FloatSlider`, `bool` → `Checkbox`, etc.
This is the closest analogue to how **dash-fn-form** works.

In [4]:
interact(make_wave);

interactive(children=(FloatSlider(value=1.0, description='amplitude', max=3.0, min=-1.0), FloatSlider(value=1.…

## What dash-fn-form does differently

| | ipywidgets `interact` | dash-fn-form `build_config` |
|---|---|---|
| Type inference | From default value type | From `typing` annotations + default |
| `Literal[...]` | Not natively supported | → `dcc.Dropdown` |
| `date` / `datetime` | Not natively supported | → `dcc.DatePickerSingle` |
| `list[T]` / `tuple` | Not natively supported | → comma-separated text input |
| Runtime defaults | Static only | `FieldHook` — live from Dash state |
| Environment | Jupyter only | Any Dash app |
| Update trigger | Live (on every change) | Explicit callback (user controls) |
| Styling | Limited | Full `styles` / `class_names` / `component_overrides` |